# Skill을 쓰는 LangGraph 에이전트

Claude Code 의 스킬 메커니즘을 LangGraph 로 재현한다. 핵심은 **progressive disclosure**:

1. 시스템 프롬프트에는 스킬의 **이름 + 설명(프론트매터)만** 넣는다 — 컨텍스트 절약
2. 에이전트가 작업과 관련 있다고 판단하면 **도구 호출로 스킬 본문을 로드**한다
3. 로드된 지침에 따라 작업을 수행한다

실험 목표: **절차형 스킬**(alarm-report — 출력 형식 규정)과 **덕목형 스킬**(karpathy-guidelines — 행동 지침)의
효과 차이를 직접 관찰한다.

> 사전 조건: Ollama 서버 + `gemma4:e4b` 모델

## 1. 스킬 검색 (discovery)

`skills/*/SKILL.md` 를 찾아 프론트매터(name, description)만 파싱한다.
본문은 아직 읽지 않는다 — 이 시점의 비용은 스킬당 한두 줄뿐이다.

In [1]:
from pathlib import Path

SKILLS_DIR = Path("../skills")


def parse_frontmatter(md_text: str) -> dict:
    """--- ... --- 블록에서 key: value 만 단순 파싱"""
    meta = {}
    parts = md_text.split("---")
    if len(parts) >= 3:
        for line in parts[1].strip().splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                meta[k.strip()] = v.strip()
    return meta


skills = {}  # name -> {description, path}
for f in SKILLS_DIR.glob("*/SKILL.md"):
    meta = parse_frontmatter(f.read_text(encoding="utf-8"))
    skills[meta["name"]] = {"description": meta["description"], "path": f}

for name, s in skills.items():
    print(f"{name}: {s['description']}")

alarm-report: 설비 알람 발생 보고서를 작성할 때 사용. 알람 코드, 조치 내용이 언급되면 이 형식을 따른다.
karpathy-guidelines: 코드를 작성하거나 리팩토링할 때 사용. 과잉 복잡화를 피하고 요청 범위를 지키는 코딩 행동 지침.


## 2. 에이전트 구성

- 시스템 프롬프트: 스킬 목록(이름+설명)과 "관련 있으면 read_skill 로 본문을 읽고 따르라"는 지시
- 도구: `read_skill(name)` — 스킬 본문 전체를 반환

In [2]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama


@tool
def read_skill(name: str) -> str:
    """스킬의 전체 지침을 읽는다. name 은 스킬 목록에 있는 이름이어야 한다."""
    if name not in skills:
        return f"오류: '{name}' 스킬 없음. 사용 가능: {list(skills)}"
    return skills[name]["path"].read_text(encoding="utf-8")


skill_list = "\n".join(f"- {name}: {s['description']}" for name, s in skills.items())

SYSTEM = f"""너는 제조 현장 지원 어시스턴트다.

사용 가능한 스킬 목록:
{skill_list}

규칙: 작업이 스킬 설명과 관련 있으면, 답하기 전에 반드시 read_skill 도구로
해당 스킬의 지침을 읽고 그대로 따라라. 관련 스킬이 없으면 그냥 답하라."""

llm = ChatOllama(model="gemma4:e4b", base_url="http://localhost:11434")

try:  # langchain v1
    from langchain.agents import create_agent
    agent = create_agent(llm, tools=[read_skill], system_prompt=SYSTEM)
except ImportError:  # 구버전 fallback
    from langgraph.prebuilt import create_react_agent
    agent = create_react_agent(llm, tools=[read_skill], prompt=SYSTEM)

print(SYSTEM)

너는 제조 현장 지원 어시스턴트다.

사용 가능한 스킬 목록:
- alarm-report: 설비 알람 발생 보고서를 작성할 때 사용. 알람 코드, 조치 내용이 언급되면 이 형식을 따른다.
- karpathy-guidelines: 코드를 작성하거나 리팩토링할 때 사용. 과잉 복잡화를 피하고 요청 범위를 지키는 코딩 행동 지침.

규칙: 작업이 스킬 설명과 관련 있으면, 답하기 전에 반드시 read_skill 도구로
해당 스킬의 지침을 읽고 그대로 따라라. 관련 스킬이 없으면 그냥 답하라.


## 3. 실행 — 절차형 스킬 (alarm-report)

알람 관련 요청 → 에이전트가 read_skill("alarm-report") 를 호출하고 형식대로 답하는지 확인.
메시지 흐름을 출력해서 **도구 호출이 실제로 일어났는지** 본다.

In [9]:
def run(question: str):
    result = agent.invoke({"messages": [("user", question)]})
    for m in result["messages"]:
        kind = m.type
        if kind == "ai" and getattr(m, "tool_calls", None):
            print(f"[{kind}] tool_calls: {[tc['name'] + str(tc['args']) for tc in m.tool_calls]}")
        else:
            content = m.content if isinstance(m.content, str) else str(m.content)
            print(f"[{kind}] {content}")
        print("-" * 60)
    return result


_ = run("CX-200 컨베이어에서 E-04 알람이 떴어. 냉각팬 필터 청소하고 복구했는데 보고서 좀 써줘.")

[human] CX-200 컨베이어에서 E-04 알람이 떴어. 냉각팬 필터 청소하고 복구했는데 보고서 좀 써줘.
------------------------------------------------------------
[ai] tool_calls: ["read_skill{'name': 'alarm-report'}"]
------------------------------------------------------------
[tool] ---
name: alarm-report
description: 설비 알람 발생 보고서를 작성할 때 사용. 알람 코드, 조치 내용이 언급되면 이 형식을 따른다.
---

# 설비 알람 보고서 작성 규칙

알람 관련 보고서를 요청받으면 반드시 아래 형식을 정확히 따른다.

## 출력 형식

```
[설비 알람 보고서]
■ 설비명: (설비 이름과 모델)
■ 알람코드: (코드) — (알람 의미)
■ 발생상황: (언제, 어떤 조건에서 발생했는지 한 문장)
■ 원인분석: (추정 원인, 번호 목록)
■ 조치내용: (수행한/수행할 조치, 번호 목록)
■ 재발방지: (예방 대책 한 가지 이상)
■ 후속조치 필요여부: 필요 / 불필요 중 하나
```

## 규칙

- 모든 항목을 빠짐없이 채운다. 정보가 없으면 "(정보 없음 — 확인 필요)"로 표기한다.
- 원인분석과 조치내용은 번호 목록으로 쓴다.
- 추측인 내용에는 문장 끝에 "(추정)"을 붙인다.
- 형식 외의 인사말, 부연 설명을 덧붙이지 않는다.

------------------------------------------------------------
[ai] [설비 알람 보고서]
■ 설비명: CX-200 컨베이어
■ 알람코드: E-04 — (정보 없음 — 확인 필요)
■ 발생상황: (정보 없음 — 확인 필요)
■ 원인분석:
1. 냉각팬 필터 오염으로 인한 성능 저하(추정)
■ 조치내용:
1. 냉각팬 필터 청소 완료
2. 설비 복구 및 정상 가동 테스트 실시
■ 재발방지: 주기

## 4. 대조 실험 — 스킬 없이 같은 질문

스킬 목록이 없는 순수 LLM 에 같은 질문 → 형식이 제각각인 답이 나오는 것을 확인.
**절차형 스킬의 효과는 이렇게 눈에 보인다.**

In [4]:
resp = llm.invoke("CX-200 컨베이어에서 E-04 알람이 떴어. 냉각팬 필터 청소하고 복구했는데 보고서 좀 써줘.")
print(resp.content)

요청하신 내용을 기반으로 상황을 공식적으로 기록하고 상사나 관련 부서에 보고할 수 있는 세 가지 버전의 보고서를 작성해 드립니다. 현장의 분위기나 받는 사람에 따라 적절한 버전을 선택하여 사용하시면 됩니다.

---

## 📋 옵션 1: 표준 공식 보고서 (가장 추천)
*(문서 파일 형태로 제출하거나, 매우 격식을 차려야 하는 경우에 적합합니다.)*

**[제목] CX-200 컨베이어 E-04 알람 발생 및 조치 완료 보고서**

| 작성일 | 202X. XX. XX. |
| :--- | :--- |
| 장비명/모델명 | CX-200 컨베이어 시스템 |
| 담당자 | [작성자 이름] |
| 발생 일시 | YYYY년 MM월 DD일 HH:MM경 |
| 조치 완료 일시 | YYYY년 MM월 DD일 HH:MM경 |

---

**1. 현상 및 알람 내용 (Issue Description)**

*   **발생 알람 코드:** E-04
*   **증상:** CX-200 컨베이어 작동 중 [특정 부위/섹션]에서 E-04 에러 코드가 발생하며 시스템이 정지함. (※E-04는 보통 과열 또는 냉각 시스템 이상을 의미합니다.)
*   **초기 진단:** 알람 메시지를 확인하고, 해당 코드와 연관된 냉각 시스템(Cooling Fan)의 성능 저하가 원인으로 추정됨.

**2. 원인 분석 (Root Cause Analysis)**

*   현장 점검 결과, 컨베이어 장비에 설치된 **냉각팬 필터(Cooling Fan Filter)**에 이물질 및 먼지 축적이 심하여 공기 흐름이 현저히 저하되었음.
*   필터 막힘으로 인해 냉각팬의 효율적인 작동이 불가능해졌고, 시스템 과열을 방지하기 위해 E-04 알람이 발생한 것으로 최종 원인을 파악함.

**3. 조치 사항 (Action Taken)**

1.  **안전 확보:** 작업 구역 접근 통제 및 전원 차단(LOTO 절차 준수)을 실시하여 안전사고를 예방함.
2.  **필터 청소:** 냉각팬 필터를 분리하고, 흡입된 먼지 및 오염 

## 5. 대조 실험 2 — 덕목형 스킬 (karpathy-guidelines)

이번엔 코딩 요청. 스킬 있는 에이전트 vs 순수 LLM 의 코드를 비교해보자.

관찰 포인트: alarm-report 만큼 차이가 뚜렷한가? — "형식을 규정하는 절차형 스킬"과 달리
"태도를 규정하는 덕목형 스킬"은 효과가 묽거나 아예 안 보일 수 있다. 직접 판단해볼 것.

In [10]:
coding_q = "파이썬으로 숫자 리스트의 평균을 구하는 함수 만들어줘"

print("===== 스킬 에이전트 =====")
_ = run(coding_q)

print("\n===== 순수 LLM =====")
print(llm.invoke(coding_q).content)

===== 스킬 에이전트 =====
[human] 파이썬으로 숫자 리스트의 평균을 구하는 함수 만들어줘
------------------------------------------------------------
[ai] 요청하신 대로 파이썬으로 숫자 리스트의 평균을 구하는 함수를 만들어 드릴게요.

가장 간단하고 효율적인 방법은 리스트의 합계를 요소 개수로 나누는 것입니다.

### 📊 Python 코드 예시

```python
def calculate_average(number_list):
    """
    주어진 숫자 리스트의 평균을 계산하여 반환합니다.
    리스트가 비어있으면 0 또는 적절한 오류 메시지를 반환할 수 있습니다. (여기서는 0을 반환)
    """
    # 리스트의 길이가 0인지 확인하여 ZeroDivisionError 방지
    if not number_list:
        print("경고: 입력된 리스트가 비어있습니다. 평균을 계산할 수 없습니다.")
        return 0
    
    # sum() 함수로 모든 숫자의 합계를 구하고, len()으로 요소 개수를 얻어 나눕니다.
    average = sum(number_list) / len(number_list)
    return average

# --- 사용 예시 ---

# 1. 일반적인 숫자 리스트
data1 = [10, 20, 30, 40, 50]
avg1 = calculate_average(data1)
print(f"리스트 {data1}의 평균: {avg1}") # 예상 출력: 30.0

# 2. 소수점 숫자가 포함된 리스트
data2 = [1.5, 2.5, 3.0]
avg2 = calculate_average(data2)
print(f"리스트 {data2}의 평균: {avg2}") # 예상 출력: 7.0 / 3 = 2.333...

# 3. 빈 리스트 (에러 처리 테스트)
data_empty = []
avg_empty = calculate_

## 6. (선택) Langfuse 트레이싱

에이전트 루프(스킬 로드 → 재추론 → 최종 답)가 trace 트리로 어떻게 보이는지 확인.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

ENV_PATH = Path(r"{CHANGEME}\langfuse\.env")
assert ENV_PATH.exists(), f".env 경로를 먼저 수정하세요: {ENV_PATH}"
load_dotenv(ENV_PATH)

from langfuse import get_client
from langfuse.langchain import CallbackHandler

handler = CallbackHandler()
result = agent.invoke(
    {"messages": [("user", "E-04 알람 보고서 작성해줘. 원인은 아직 몰라.")]},
    config={"callbacks": [handler]},
)
get_client().flush()
print(result["messages"][-1].content)

```
[설비 알람 보고서]
■ 설비명: (정보 없음 — 확인 필요)
■ 알람코드: E-04 — (알람 의미 정보 없음 — 확인 필요)
■ 발생상황: (시간 및 상황에 대한 구체적인 내용이 없습니다. 즉시 현장 확인 필요.)
■ 원인분석: 
1. 현재까지 파악된 명확한 원인은 없습니다.
2. 전원 불안정 또는 센서 오작동 가능성 (추정) — 상세 점검 필요
■ 조치내용: 
1. 설비 작동 일시 정지 및 안전 확보.
2. 알람 발생 직전의 운전 조건을 확인하였습니다.
3. 현장 담당자 호출하여 재점검을 준비했습니다.
■ 재발방지: (정보 없음 — 해당 코드에 대한 표준 운영 절차(SOP) 점검 필요)
■ 후속조치 필요여부: 필요
```

**※ 참고 사항:** 요청하신 알람 보고서 양식에 따라 작성되었으나, 설비명 및 구체적인 발생 상황 정보가 없어 주요 항목은 "(정보 없음 — 확인 필요)"로 표기했습니다. 해당 정보를 확보하여 추가해주시면 보다 정확한 보고서 작성이 가능합니다.
